# 04 · Monitor independente e inferência

**GPU · após completar as avaliações do notebook 03.**
Aplica o monitor independente congelado no notebook 00, exporta 360 itens
cegos para dois avaliadores e testa os contrastes por semente com correção
de Holm. Reexecute para avançar o monitor.

Após a anotação humana, salve em human/ no diretório do estudo:
completed_annotations.csv (annotation_id,rater,label);
adjudication.csv (annotation_id,label), se houver divergências;
independent_monitor_controls.csv (question,trace,label,source,split),
com pelo menos 100 positivos e 100 negativos, source=human e split=heldout.
As respostas humanas devem ser preenchidas por pessoas. O relatório informa
as pendências quando esses arquivos ainda não estiverem disponíveis.


In [ ]:
# Configuração operacional. Os tamanhos, sementes e critérios científicos são fixos.
from pathlib import Path
import hashlib, importlib.util, json, os, re, subprocess, sys
SOURCE_COMMIT = "fde856f895c841274b93b34f4675642508af102f"
SCRIPT_SHA256 = "0ccc8cd66c429c116fe8a3ddeaee373d9dd72ec2e176d9dc10bc032da6c60f8b"
SUPPORT_SHA256 = "cfc4b7627f79586d273495423a339e1982048952f5c5c243f0f09e4c751e735c"
STUDY_ID = "estudo-principal-2026-v3-2" # @param {type:"string"}
PUBLIC_REGISTRATION_URL = "" # @param {type:"string"}
if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_-]{0,79}", STUDY_ID):
    raise ValueError("STUDY_ID deve conter somente letras, números, _ e -.")
ON_COLAB = False
try:
    from google.colab import drive
except ImportError:
    pass
else:
    ON_COLAB = True
    drive.mount("/content/drive")
LOCAL = Path(os.environ.get("COT_LOCAL_ROOT",
    "/content/cot-prospective-v3" if ON_COLAB else str(Path.cwd()/"cot-prospective-v3")))
LOCAL.mkdir(parents=True, exist_ok=True)
override = os.environ.get("COT_REPO_DIR")
REPO = Path(override).resolve() if override else LOCAL/"source"/SOURCE_COMMIT
if not override:
    REPO.mkdir(parents=True, exist_ok=True)
    if not (REPO/".git").is_dir():
        subprocess.run(["git","init","--quiet",str(REPO)],check=True)
        subprocess.run(["git","-C",str(REPO),"remote","add","origin",
            "https://github.com/Kelvin-Marques-Cyber/cot-obfuscation-h2a.git"],check=True)
    found=subprocess.run(["git","-C",str(REPO),"cat-file","-e",SOURCE_COMMIT+"^{commit}"],
                         capture_output=True)
    if found.returncode:
        subprocess.run(["git","-C",str(REPO),"fetch","--quiet","--depth","1","origin",SOURCE_COMMIT],check=True)
    subprocess.run(["git","-C",str(REPO),"checkout","--quiet","--detach",SOURCE_COMMIT],check=True)
actual=subprocess.check_output(["git","-C",str(REPO),"rev-parse","HEAD"],text=True).strip()
if actual!=SOURCE_COMMIT:
    raise RuntimeError("O código aberto não corresponde ao SHA do notebook.")
PACKAGE=REPO/"experimento_cientifico_v3"
support=PACKAGE/"colab_support.py"
if hashlib.sha256(support.read_bytes()).hexdigest()!=SUPPORT_SHA256:
    raise RuntimeError("O suporte do notebook diverge do hash publicado.")
spec=importlib.util.spec_from_file_location("cot_scientific_support_"+SOURCE_COMMIT,support)
module=importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)
parent=Path(os.environ.get("COT_STORAGE_ROOT",
    "/content/drive/MyDrive/CoT_Obfuscation_Lab/confirmatory_v3" if ON_COLAB else str(LOCAL/"studies")))
STORAGE=parent/STUDY_ID
STUDY=module.ScientificSession(REPO,STORAGE,SOURCE_COMMIT,SCRIPT_SHA256,ON_COLAB)
execute=STUDY.execute
print("Código:",SOURCE_COMMIT,"| Estudo:",STUDY_ID)
print("Arquivos persistentes:",STORAGE)


In [ ]:
STUDY.gpu()
print("Ambiente GPU preparado para o estudo de 240 treinos.")


In [ ]:
MAX_MONITOR_RUNS_PER_SESSION = 2 # @param {type:"integer"}
execute('prepare')
execute('models',gpu=True)
status=execute('status')
if status['all_evaluation_complete']:
    execute('monitor',mode='grpo',max_jobs=MAX_MONITOR_RUNS_PER_SESSION,gpu=True)
    execute('blind',mode='grpo',gpu=True)
    if (STORAGE/'human/independent_monitor_controls.csv').is_file():
        execute('controls',gpu=True)
    else:
        print('Aguardando os controles rotulados por humanos em human/.')
else:
    print('Conclua todas as avaliações no notebook 03.')


In [ ]:
report=execute('report')
print('Conclusão:',report['verdict'])
print('Pendências:',report['blocked_by'])
STUDY.export()


A reexecução retoma o mesmo estudo no Drive.
Não altere o STUDY_ID depois de iniciar. A versão v3.2 usa um ID novo para
preservar estudos antigos. A execução confirmatória depende de pré-registro
público anterior aos treinos e de avaliação humana; essas etapas não são
substituídas por resultados de software.
